use the cells below this one merges SVM AND RF only

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# # Load datasets
# X = pd.read_excel('../minmax.xlsx')
# y = pd.read_csv('../idC_with_header.csv')['Label']

# # Split dataset
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Define fitness function
# def fitness_function(params, model_type='svm'):
#     if model_type == 'svm':
#         C, gamma = params
#         model = SVC(C=C, gamma=gamma)
#     elif model_type == 'rf':
#         n_estimators, max_depth = params
#         model = RandomForestClassifier(n_estimators=int(n_estimators), max_depth=int(max_depth))
#     else:
#         raise ValueError("Unsupported model type.")
#     model.fit(X_train, y_train)
#     predictions = model.predict(X_test)
#     return accuracy_score(y_test, predictions)

# # GSA parameters
# num_agents = 10
# max_iter = 20
# G0 = 100
# alpha = 20

# # Hyperparameter bounds
# svm_bounds = {'C': (0.1, 100), 'gamma': (0.0001, 1)}
# rf_bounds = {'n_estimators': (10, 200), 'max_depth': (1, 50)}

# # Initialize agents
# def initialize_agents(bounds):
#     agents = []
#     for _ in range(num_agents):
#         agent = []
#         for key in bounds:
#             low, high = bounds[key]
#             agent.append(np.random.uniform(low, high))
#         agents.append(agent)
#     return np.array(agents)

# # GSA optimization
# def gsa_optimization(bounds, model_type='svm'):
#     agents = initialize_agents(bounds)
#     velocities = np.zeros_like(agents)
#     best_agent = None
#     best_fitness = -np.inf

#     for t in range(max_iter):
#         fitness = np.array([fitness_function(agent, model_type) for agent in agents])
#         best_idx = np.argmax(fitness)
#         if fitness[best_idx] > best_fitness:
#             best_fitness = fitness[best_idx]
#             best_agent = agents[best_idx].copy()

#         worst = np.min(fitness)
#         best = np.max(fitness)
#         mass = (fitness - worst) / (best - worst + 1e-10)
#         mass = mass / mass.sum()

#         G = G0 * np.exp(-alpha * t / max_iter)

#         for i in range(num_agents):
#             force = np.zeros(agents.shape[1])
#             for j in range(num_agents):
#                 if i != j:
#                     distance = np.linalg.norm(agents[j] - agents[i]) + 1e-10
#                     force += np.random.rand() * G * (mass[i] * mass[j]) * (agents[j] - agents[i]) / distance
#             acceleration = force / (mass[i] + 1e-10)
#             velocities[i] = np.random.rand() * velocities[i] + acceleration
#             agents[i] += velocities[i]

#             # Ensure agents stay within bounds
#             for k, key in enumerate(bounds):
#                 low, high = bounds[key]
#                 agents[i][k] = np.clip(agents[i][k], low, high)

#     return best_agent

# # Optimize SVM
# best_svm_params = gsa_optimization(svm_bounds, model_type='svm')
# best_C, best_gamma = best_svm_params
# svm_model = SVC(C=best_C, gamma=best_gamma)
# svm_model.fit(X_train, y_train)
# svm_predictions = svm_model.predict(X_test)




# # Evaluate SVM
# print("SVM Performance:")
# print("Accuracy:", accuracy_score(y_test, svm_predictions))
# print("Precision:", precision_score(y_test, svm_predictions, average='weighted'))
# print("Recall:", recall_score(y_test, svm_predictions, average='weighted'))
# print("F1 Score:", f1_score(y_test, svm_predictions, average='weighted'))
# print("Best Hyperparameters:", {'C': best_C, 'gamma': best_gamma})

# # Optimize RF
# best_rf_params = gsa_optimization(rf_bounds, model_type='rf')
# best_n_estimators, best_max_depth = best_rf_params
# rf_model = RandomForestClassifier(n_estimators=int(best_n_estimators), max_depth=int(best_max_depth))
# rf_model.fit(X_train, y_train)
# rf_predictions = rf_model.predict(X_test)

# # Evaluate RF
# print("\nRandom Forest Performance:")
# print("Accuracy:", accuracy_score(y_test, rf_predictions))
# print("Precision:", precision_score(y_test, rf_predictions, average='weighted'))
# print("Recall:", recall_score(y_test, rf_predictions, average='weighted'))
# print("F1 Score:", f1_score(y_test, rf_predictions, average='weighted'))
# print("Best Hyperparameters:", {'n_estimators': int(best_n_estimators), 'max_depth': int(best_max_depth)})


SVM Performance:
Accuracy: 0.9775280898876404
Precision: 0.9802568218298554
Recall: 0.9775280898876404
F1 Score: 0.9775846861594938
Best Hyperparameters: {'C': 15.430558197392699, 'gamma': 0.0044894202168458824}

Random Forest Performance:
Accuracy: 0.9325842696629213
Precision: 0.9335819721146885
Recall: 0.9325842696629213
F1 Score: 0.9298695215903967
Best Hyperparameters: {'n_estimators': 106, 'max_depth': 49}


GSA + RF Hyperparameter optimization

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
# Load dataset
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Fitness function for Random Forest ===
def fitness_function_rf(params):
    n_estimators = int(params[0])
    max_depth = int(params[1])
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return accuracy_score(y_test, predictions)

# === Initialize GSA parameters ===
num_agents = 10
max_iter = 20
G0 = 100
alpha = 20

# === Hyperparameter bounds for RF ===
bounds_rf = [(10, 200), (1, 50)]  # n_estimators, max_depth

# === Initialize agents ===
def initialize_agents(bounds):
    return np.array([[np.random.uniform(low, high) for (low, high) in bounds] for _ in range(num_agents)])

# === GSA Optimizer for Random Forest ===
def gsa_rf():
    agents = initialize_agents(bounds_rf)
    velocities = np.zeros_like(agents)
    best_agent = None
    best_fitness = -np.inf

    for t in range(max_iter):
        fitness = np.array([fitness_function_rf(agent) for agent in agents])
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_fitness:
            best_fitness = fitness[best_idx]
            best_agent = agents[best_idx].copy()

        worst = np.min(fitness)
        best = np.max(fitness)
        mass = (fitness - worst) / (best - worst + 1e-10)
        mass = mass / mass.sum()

        G = G0 * np.exp(-alpha * t / max_iter)

        for i in range(num_agents):
            force = np.zeros(agents.shape[1])
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(agents[j] - agents[i]) + 1e-10
                    force += np.random.rand() * G * (mass[i] * mass[j]) * (agents[j] - agents[i]) / distance
            acceleration = force / (mass[i] + 1e-10)
            velocities[i] = np.random.rand() * velocities[i] + acceleration
            agents[i] += velocities[i]

            # Clamp to bounds
            for k in range(len(bounds_rf)):
                low, high = bounds_rf[k]
                agents[i][k] = np.clip(agents[i][k], low, high)

    return best_agent

# === Run GSA and evaluate final model ===
best_params = gsa_rf()
best_n_estimators, best_max_depth = int(best_params[0]), int(best_params[1])

# Train final model with RF best parameters
final_model = RandomForestClassifier(n_estimators=best_n_estimators, max_depth=best_max_depth, random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# SVM Classifier
# final_model_svm = SVC(C=1.0, kernel='rbf', random_state=42)  # Default SVM parameters
# final_model_svm.fit(X_train, y_train)
# y_pred = final_model_svm.predict(X_test)

# NBC Classifier
# final_model_nb = GaussianNB()
# final_model_nb.fit(X_train, y_train)
# y_pred = final_model_nb.predict(X_test)



print(f"Best Hyperparameters: n_estimators={best_n_estimators}, max_depth={best_max_depth}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Best Hyperparameters: n_estimators=141, max_depth=25
Accuracy: 87.64%

Classification Report:
              precision    recall  f1-score   support

           1       1.00      0.50      0.67         2
           2       1.00      0.88      0.93         8
           3       0.67      1.00      0.80         6
           4       1.00      0.83      0.91        18
           5       0.82      1.00      0.90         9
           6       0.83      0.83      0.83         6
           7       1.00      1.00      1.00         1
           9       0.93      1.00      0.96        13
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         5
          12       0.56      0.83      0.67         6
          13       1.00      0.57      0.73         7
          14       1.00      0.86      0.92         7

    accuracy                           0.88        89
   macro avg       0.91      0.87      0.87        89
weighted avg       0.91      0.88      0

GSA + SVM Hyperparameter optimization

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Load dataset
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Fitness function for SVM ===
def fitness_function_svm(params):
    C = params[0]
    gamma = params[1]
    model = SVC(C=C, gamma=gamma)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return accuracy_score(y_test, predictions)


# === Initialize GSA parameters ===
num_agents = 10
max_iter = 20
G0 = 100
alpha = 20
# === Hyperparameter bounds for SVM ===
bounds_svm = [(0.1, 100), (0.0001, 1)]  # C, gamma


# === Initialize agents ===
def initialize_agents(bounds):
    return np.array([[np.random.uniform(low, high) for (low, high) in bounds] for _ in range(num_agents)])

# === GSA Optimizer for SVM ===
def gsa_svm():
    agents = initialize_agents(bounds_svm)
    velocities = np.zeros_like(agents)
    best_agent = None
    best_fitness = -np.inf

    for t in range(max_iter):
        fitness = np.array([fitness_function_svm(agent) for agent in agents])
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_fitness:
            best_fitness = fitness[best_idx]
            best_agent = agents[best_idx].copy()

        worst = np.min(fitness)
        best = np.max(fitness)
        mass = (fitness - worst) / (best - worst + 1e-10)
        mass = mass / mass.sum()

        G = G0 * np.exp(-alpha * t / max_iter)

        for i in range(num_agents):
            force = np.zeros(agents.shape[1])
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(agents[j] - agents[i]) + 1e-10
                    force += np.random.rand() * G * (mass[i] * mass[j]) * (agents[j] - agents[i]) / distance
            acceleration = force / (mass[i] + 1e-10)
            velocities[i] = np.random.rand() * velocities[i] + acceleration
            agents[i] += velocities[i]

            # Clamp to bounds
            for k in range(len(bounds_svm)):
                low, high = bounds_svm[k]
                agents[i][k] = np.clip(agents[i][k], low, high)

    return best_agent

# === Run GSA and evaluate SVM model ===
best_svm_params = gsa_svm()
best_C, best_gamma = best_svm_params[0], best_svm_params[1]



#     #SVM Classifier
    # svm_model = SVC(C=best_C, gamma=best_gamma)
    # svm_model.fit(X_train, y_train)
    # svm_pred = svm_model.predict(X_test)


    # print("\nSVM Performance:")
    # print(f"Best Hyperparameters: C={best_C:.4f}, gamma={best_gamma:.4f}")
    # print(f"Accuracy: {accuracy_score(y_test, svm_pred) * 100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(y_test, svm_pred))

""""
# Random Forest Classifier
    # Using optimized hyperparameters from SVM fitness function (best_C for n_estimators)
    # Ensure that best_C is valid for n_estimators (e.g., must be a reasonable integer)
    # rf_model = RandomForestClassifier(n_estimators=int(best_C), max_depth=10, random_state=42)  # Fixed max_depth for RF
    # rf_model.fit(X_train, y_train)
    # rf_pred = rf_model.predict(X_test)

    print("\nRandom Forest Performance (using optimized parameters from SVM fitness):")
    print(f"Best Hyperparameters for RF: n_estimators={int(best_C)}, max_depth=10")
    print(f"Accuracy: {accuracy_score(y_test, rf_pred) * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, rf_pred))
"""

#  Naive Bayes Classifier 
    # nb_model = GaussianNB()
    # nb_model.fit(X_train, y_train)
    # nb_pred = nb_model.predict(X_test)

    # print("\nNaive Bayes Performance:")
    # print(f"Accuracy: {accuracy_score(y_test, nb_pred) * 100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(y_test, nb_pred))




Naive Bayes Performance:
Accuracy: 87.64%

Classification Report:
              precision    recall  f1-score   support

           1       1.00      0.50      0.67         2
           2       1.00      0.88      0.93         8
           3       0.67      1.00      0.80         6
           4       1.00      0.83      0.91        18
           5       0.82      1.00      0.90         9
           6       0.83      0.83      0.83         6
           7       1.00      1.00      1.00         1
           9       0.93      1.00      0.96        13
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         5
          12       0.56      0.83      0.67         6
          13       1.00      0.57      0.73         7
          14       1.00      0.86      0.92         7

    accuracy                           0.88        89
   macro avg       0.91      0.87      0.87        89
weighted avg       0.91      0.88      0.88        89



GSA-Based Feature Selection for Naive Bayes Classification

In [17]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Load the dataset
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for GSA
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i in range(len(position)) if position[i] > 0.5]
    if len(selected_features) == 0:  # Avoid empty feature subsets
        return 0
    
    # Train a GaussianNB model
    model = GaussianNB()
    model.fit(X_train.iloc[:, selected_features], y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test.iloc[:, selected_features])
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize GSA parameters
num_agents = 20
num_features = X.shape[1]  # Number of features in the dataset
max_iterations = 25
G = 100  # Gravitational constant
epsilon = 1e-5  # Small value to avoid division by zero

# Initialize agents' positions randomly (binary positions for feature selection)
agents = np.random.randint(0, 2, size=(num_agents, num_features))
velocities = np.zeros_like(agents)

# GSA algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(agent) for agent in agents])
    
    # Calculate masses based on fitness
    masses = (fitness - fitness.min()) / (fitness.max() - fitness.min() + epsilon)
    masses /= masses.sum()
    
    # Update positions and velocities
    for i in range(num_agents):
        force = np.zeros(num_features)
        for j in range(num_agents):
            if i != j:
                distance = np.linalg.norm(agents[i] - agents[j]) + epsilon
                force += G * masses[j] * (agents[j] - agents[i]) / distance
        acceleration = force / (masses[i] + epsilon)
        velocities[i] = np.random.rand() * velocities[i] + acceleration
        agents[i] = agents[i] + velocities[i]
        agents[i] = np.clip(agents[i], 0, 1)
        agents[i] = np.round(agents[i])  # Convert to binary

# Evaluate the best solution
best_agent = agents[np.argmax(fitness)]
selected_features = [i for i in range(len(best_agent)) if best_agent[i] > 0.5]

if len(selected_features) == 0:
    print("No features selected by GSA. Using all features as fallback.")
    selected_features = list(range(X.shape[1]))  # fallback: use all features


#Train the final model with GaussianNB using the selected features

final_model = GaussianNB()
final_model.fit(X_train.iloc[:, selected_features], y_train)
y_pred = final_model.predict(X_test.iloc[:, selected_features])

# Print performance metrics
print(f"Selected Features: {selected_features}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# Train the final SVM model using the selected features

    # svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
    # svm_model.fit(X_train.iloc[:, selected_features], y_train)
    # svm_y_pred = svm_model.predict(X_test.iloc[:, selected_features])

    # # Print SVM performance metrics
    # print("\n--- SVM Classifier Performance ---")
    # print(f"SVM Accuracy: {accuracy_score(y_test, svm_y_pred) * 100:.2f}%")
    # print("SVM Classification Report:")
    # print(classification_report(y_test, svm_y_pred))
    
    
# Train the final RF model using the selected features
# rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
# rf_model.fit(X_train.iloc[:, selected_features], y_train)
# rf_y_pred = rf_model.predict(X_test.iloc[:, selected_features])

#     # Print RF performance metrics
# print("\n--- Random Forest Classifier Performance ---")
# print(f"RF Accuracy: {accuracy_score(y_test, rf_y_pred) * 100:.2f}%")
# print("RF Classification Report:")
# print(classification_report(y_test, rf_y_pred))

Selected Features: [1, 4, 5, 6, 8, 11, 12, 13, 15, 16, 17, 18, 19, 20, 23, 29, 30, 31, 33, 37, 39, 40, 41, 42, 44, 45, 48, 50, 53, 54, 56, 62, 71, 72, 76, 77, 86, 88, 90, 91, 92, 93, 94, 95, 97, 99, 100, 101, 102, 103, 104, 105, 107, 110, 111, 115, 116, 118, 119, 120, 121, 122, 125, 126, 127, 128, 131, 133, 135, 137, 139, 142, 143, 144, 145, 146, 147, 149, 152, 153, 155, 158, 159, 161, 165, 167, 169, 170, 171, 172, 174, 176, 177, 178, 179, 180, 184, 185, 186, 187, 190, 191, 192, 194, 195, 198, 202, 203, 204, 205, 206, 207, 208, 213, 214, 215, 218, 220, 221, 223, 224, 226, 228, 229, 234, 235, 237, 238, 239, 240, 241, 242, 243, 246, 249, 250, 252, 254, 255, 260, 265, 268, 275, 276, 277, 282, 283, 284, 285, 286, 287, 290, 291, 292, 294, 296, 297, 300, 302, 304, 306, 307, 308, 312, 313, 315, 316, 319, 323, 324, 326, 327, 329, 330, 331, 333, 338, 339, 340, 341, 343, 344, 347, 353, 354, 356, 360, 362, 363, 366, 367, 368, 370, 371, 373, 374, 377, 379, 380, 382, 383, 384, 386, 392, 394, 397, 3